# AI Music OS — Kokoro TTS v7 — Expanded

Free local models; no API key. Choose **T4 GPU**, then **Run all** in a fresh Colab session. Open the printed Gradio link. Each notebook uses its own pinned Python 3.11 environment, leaving Colab's Python 3.13 packages untouched. Models and WAV outputs remain on Drive. Keep the notebook session open; free Colab can disconnect it.

Upload this notebook directly to Colab, or replace its matching file in GitHub. No other repository files are required. Dependency and CPU import tests passed locally; Colab T4 inference is not certified.

**Setup revision V6:** does not use Colab system venv/ensurepip. Replace all three notebooks, not just the voice-clone file.


In [ ]:
MODULE = 'kokoro'
# Separate Python 3.11 environment; do not install AI packages into Colab's kernel.
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, time, re
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI_Music_OS')
CACHE = ROOT / 'cache' / 'huggingface'
OUT = ROOT / 'outputs' / {'ace': 'ace_step', 'kokoro': 'kokoro', 'f5': 'qwen_voice'}[MODULE]
VOICE_BANK = ROOT / 'voice_bank'
MODELS = ROOT / 'models' / 'ace_step'
for p in (CACHE, OUT, VOICE_BANK, MODELS): p.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV['HF_HOME'] = str(CACHE)
ENV['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'hub')
ENV.pop('TRANSFORMERS_CACHE', None)
ENV['PYTHONUNBUFFERED'] = '1'
# Prevent Colab kernel paths from leaking into the isolated interpreter.
ENV.pop('PYTHONPATH', None)
ENV.pop('PYTHONHOME', None)
ENV['PYTHONNOUSERSITE'] = '1'
WORK = Path('/content/ai_music_v6') / MODULE
WORK.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'install_v6.log'
def run_logged(cmd, cwd=None):
    with LOG.open('a') as log:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError(f'Command failed ({result.returncode}). Complete output: {LOG}')
# Colab's system Python can fail at venv/ensurepip. Download the official uv
# wheel with existing pip and extract only its binary, without installing it
# into the kernel or running ensurepip. uv manages the separate AI interpreter.
import zipfile, shutil
UV_HOME = Path('/content/ai_music_v6/bootstrap')
UV_HOME.mkdir(parents=True, exist_ok=True)
UV = UV_HOME / 'uv'
if not UV.exists():
    WHEELS = UV_HOME / 'wheels'
    WHEELS.mkdir(exist_ok=True)
    run_logged([sys.executable, '-m', 'pip', 'download', '--no-cache-dir',
                '--no-deps', '--only-binary=:all:', '--index-url', 'https://pypi.org/simple',
                '--dest', str(WHEELS), 'uv==0.13.8'])
    wheels = sorted(WHEELS.glob('uv-0.13.8-*.whl'))
    if len(wheels) != 1:
        raise RuntimeError(f'Expected one uv wheel, found {len(wheels)}. See {LOG}')
    temporary = UV.with_suffix('.tmp')
    try:
        with zipfile.ZipFile(wheels[0]) as archive:
            binaries = [name for name in archive.namelist() if name.endswith('.data/scripts/uv')]
            if len(binaries) != 1: raise RuntimeError('Official uv wheel has no unique Linux binary.')
            with archive.open(binaries[0]) as src, temporary.open('wb') as dst:
                shutil.copyfileobj(src, dst)
        temporary.chmod(0o755)
        temporary.replace(UV)
    finally:
        temporary.unlink(missing_ok=True)
run_logged([str(UV), '--version'])
run_logged([str(UV), 'python', 'install', '3.11.18'])
VENV = WORK / 'venv'
PY = VENV / 'bin' / 'python'
if not PY.exists(): run_logged([str(UV), 'venv', '--python', '3.11.18', str(VENV)])
print('Colab kernel:', sys.version.split()[0], '| AI runtime: Python 3.11.18')
print('Outputs:', OUT, '| Installation log:', LOG)


In [ ]:
LOCK_CONTENT = 'addict==2.4.0\naiofiles==23.2.1\nannotated-doc==0.0.6\nannotated-types==0.8.0\nanyio==4.16.0\nattrs==26.1.0\naudioread==3.1.0\nbabel==2.18.0\nblis==1.3.4\ncatalogue==2.0.10\ncertifi==2026.10.23\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.1\ncloudpathlib==0.26.0\ncloudpickle==3.1.2\nconfection==0.1.5\ncsvw==4.2.0\ncurated-tokenizers==0.0.10\ncurated-transformers==0.1.1\ncymem==2.0.16\ndecorator==5.3.1\ndlinfo==2.0.0\ndocopt==0.6.2\nen-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl\nespeakng-loader==0.2.4\nfastapi==0.162.7\nffmpy==1.0.0\nfilelock==4.2.10\nfsspec==2026.9.0\ngradio==5.22.0\ngradio-client==1.8.0\ngroovy==0.1.2\nh11==0.16.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.30.2\nidna==3.20\nisodate==0.7.2\njinja2==3.1.6\njoblib==1.6.0\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\nkokoro==0.9.4\nlangcodes==3.5.1\nlanguage-tags==1.3.2\nlazy-loader==0.6\nlibrosa==0.11.0\nllvmlite==0.50.0\nloguru==0.8.0\nmarkdown-it-py==4.3.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmisaki==0.9.4\nmpmath==1.3.0\nmsgpack==1.2.4\nmurmurhash==1.0.16\nnarwhals==2.28.0\nnetworkx==3.6.1\nnum2words==0.5.14\nnumba==0.68.1\nnumpy==1.26.4\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nopentelemetry-api==1.45.2\norjson==3.13.0\npackaging==26.3\npandas==2.3.3\nphonemizer-fork==3.3.2\npillow==11.3.0\npip==25.1.1\nplatformdirs==4.14.5\npooch==1.9.0\npreshed==3.0.15\npycparser==3.12\npydantic==2.10.6\npydantic-core==2.27.2\npydub==0.25.1\npygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\npython-multipart==0.0.33\npytz==2026.6\npyyaml==6.0.3\nrdflib==7.6.0\nreferencing==0.37.0\nregex==2026.11.9\nrequests==2.34.2\nrfc3986==1.5.0\nrich==15.0.0\nrpds-py==2026.9.1\nruff==0.17.5\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.13.1\nsegments==2.4.0\nsemantic-version==2.10.0\nsetuptools==84.2.1\nshellingham==1.5.4\nsmart-open==7.7.1\nsoundfile==0.13.1\nsoxr==1.1.0\nspacy==3.8.4\nspacy-curated-transformers==0.3.1\nspacy-legacy==3.0.12\nspacy-loggers==1.0.5\nsrsly==2.5.6\nstarlette==0.52.1\nsympy==1.13.1\ntermcolor==3.3.0\nthinc==8.3.11\nthreadpoolctl==3.8.0\ntokenizers==0.21.4\ntomlkit==0.13.3\ntorch==2.6.0+cu124\ntorchaudio==2.6.0+cu124\ntorchvision==0.21.0+cu124\ntqdm==4.70.1\ntransformers==4.50.0\ntriton==3.2.0\ntyper==0.29.1\ntyper-slim==0.24.0\ntyping-extensions==4.16.0\ntyping-inspection==0.4.5\ntzdata==2026.6\nuritemplate==4.2.0\nurllib3==2.8.0\nuvicorn==0.54.0\nwasabi==1.1.3\nweasel==0.4.3\nwebsockets==15.0.1\nwrapt==2.5.2\n'
CHECK_IMPORTS = 'import torch, torchaudio, torchvision, gradio, addict, misaki.en; from transformers import AlbertModel; from kokoro import KPipeline; print("Kokoro imports: OK")'
# Full dependency resolution from a pinned lock, including addict and phonemizers.
LOCK = WORK / 'requirements.lock'
LOCK.write_text(LOCK_CONTENT)
run_logged([str(UV), 'pip', 'sync', '--python', str(PY), '--no-cache',
            '--index', 'https://download.pytorch.org/whl/cu124',
            '--index', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match', str(LOCK)])
run_logged([str(UV), 'pip', 'check', '--python', str(PY)])
run_logged([str(PY), '-c', CHECK_IMPORTS])
print('Complete installation + imports: OK')


In [ ]:
# Verify the actual AI runtime, not Colab's preinstalled torch.
run_logged([str(PY), '-c', "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'); assert torch.cuda.is_available(), 'Select T4 GPU in Runtime > Change runtime type, then rerun.'"])
print('T4/CUDA check: OK')


In [ ]:
# Write standalone app and shared exports; both are embedded, no repository upload needed.
APP = WORK / 'app.py'
(WORK / 'audio_tools.py').write_text('"""Shared local audio exports; embedded beside each standalone notebook app."""\nfrom pathlib import Path\nimport json\nimport math\nimport subprocess\nimport time\nimport zipfile\nimport numpy as np\nimport soundfile as sf\n\n\ndef read_text_file(path):\n    if not path:\n        raise ValueError(\'Choose a UTF-8 .txt file.\')\n    p = Path(path)\n    if p.suffix.lower() != \'.txt\' or p.stat().st_size > 1_000_000:\n        raise ValueError(\'Use a .txt file smaller than 1 MB.\')\n    return p.read_text(encoding=\'utf-8-sig\')\n\n\ndef split_text(text, limit=180):\n    """Bound chunks even for punctuation-free text (Espeak otherwise truncates)."""\n    import re\n    chunks = []\n    for line in (text or \'\').strip().splitlines():\n        for part in re.split(r\'(?<=[.!?。！？।])\\s*\', line.strip()):\n            while len(part) > limit:\n                cut = part.rfind(\' \', 0, limit + 1)\n                cut = cut if cut > 0 else limit\n                chunks.append(part[:cut].strip())\n                part = part[cut:].strip()\n            if part:\n                chunks.append(part)\n    return chunks\n\n\ndef export_audio(wav, format=\'WAV\', pitch=0, normalize=False):\n    if format not in (\'WAV\', \'MP3\', \'FLAC\'):\n        raise ValueError(\'Choose WAV, MP3 or FLAC.\')\n    if not math.isfinite(float(pitch)) or not -6 <= float(pitch) <= 6:\n        raise ValueError(\'Pitch must be between -6 and +6 semitones.\')\n    wav = Path(wav)\n    if format == \'WAV\' and not pitch and not normalize:\n        return str(wav)\n    target = wav.with_name(wav.stem + \'_export.\' + format.lower())\n    filters = []\n    if pitch:\n        ratio = 2 ** (float(pitch) / 12)\n        filters += [f\'asetrate={sf.info(wav).samplerate}*{ratio}\', \'aresample=24000\', f\'atempo={1 / ratio}\']\n    if normalize:\n        filters.append(\'loudnorm=I=-16:TP=-1.5:LRA=11\')\n    cmd = [\'ffmpeg\', \'-y\', \'-v\', \'error\', \'-i\', str(wav)]\n    if filters:\n        cmd += [\'-af\', \',\'.join(filters)]\n    if format == \'MP3\':\n        cmd += [\'-codec:a\', \'libmp3lame\', \'-b:a\', \'192k\']\n    subprocess.run(cmd + [str(target)], check=True)\n    return str(target)\n\n\ndef save_metadata(wav, data):\n    Path(wav).with_suffix(\'.json\').write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef package_files(files, out, prefix):\n    path = Path(out) / f\'{prefix}_{time.time_ns()}.zip\'\n    with zipfile.ZipFile(path, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for f in files:\n            p = Path(f)\n            archive.write(p, p.name)\n            for suffix in (\'.json\', \'.srt\'):\n                sidecar = p.with_suffix(suffix)\n                if sidecar.exists():\n                    archive.write(sidecar, sidecar.name)\n    return str(path)\n\n\ndef recent_outputs(out):\n    files = sorted((p for p in Path(out).iterdir() if p.suffix.lower() in (\'.wav\', \'.mp3\', \'.flac\', \'.zip\')), key=lambda p: p.stat().st_mtime, reverse=True)[:30]\n    return [str(p) for p in files]\n\n\ndef timestamp(seconds):\n    ms = round(seconds * 1000)\n    hours, ms = divmod(ms, 3600000)\n    minutes, ms = divmod(ms, 60000)\n    sec, ms = divmod(ms, 1000)\n    return f\'{hours:02}:{minutes:02}:{sec:02},{ms:03}\'\n', encoding='utf-8')
APP.write_text("from pathlib import Path\nimport os\nimport time\nimport numpy as np\nimport soundfile as sf\nimport gradio as gr\nfrom kokoro import KPipeline\nfrom audio_tools import split_text, export_audio, save_metadata, package_files, read_text_file, recent_outputs, timestamp\n\nOUT = Path(os.environ['AI_MUSIC_OUT'])\nOUT.mkdir(parents=True, exist_ok=True)\nLANGUAGES = {'American English':'a', 'British English':'b', 'Hindi':'h', 'Spanish':'e', 'French':'f', 'Italian':'i', 'Brazilian Portuguese':'p'}\nVOICE_CHOICES = 'af_alloy af_aoede af_bella af_heart af_jessica af_kore af_nicole af_nova af_river af_sarah af_sky am_adam am_echo am_eric am_fenrir am_liam am_michael am_onyx am_puck am_santa bf_alice bf_emma bf_isabella bf_lily bm_daniel bm_fable bm_george bm_lewis ef_dora em_alex em_santa ff_siwis hf_alpha hf_beta hm_omega hm_psi if_sara im_nicola pf_dora pm_alex pm_santa'.split()\nSAMPLE_RATE = 24000\npipelines = {}\n\n\ndef get_pipeline(code):\n    if code not in pipelines:\n        if pipelines:\n            pipelines[code] = KPipeline(lang_code=code, model=next(iter(pipelines.values())).model)\n        else:\n            pipelines[code] = KPipeline(lang_code=code)\n    return pipelines[code]\n\n\ndef voices_for_language(language):\n    code = LANGUAGES[language]\n    return [v for v in VOICE_CHOICES if v.startswith(code)]\n\n\ndef change_language(language):\n    choices = voices_for_language(language)\n    return gr.update(choices=choices, value=choices[0]), gr.update(choices=['None'] + choices, value='None')\n\n\ndef generate_audio(text, voice='af_heart', speed=1.0, pause=0.15, blend_voice='None', blend=0.5, format='WAV', pitch=0, normalize=False):\n    text = (text or '').strip()\n    if not text:\n        raise ValueError('Text is empty.')\n    if voice not in VOICE_CHOICES:\n        raise ValueError(f'Unsupported voice: {voice}')\n    if not 0.5 <= float(speed) <= 2 or not 0 <= float(pause) <= 2:\n        raise ValueError('Speed must be 0.5–2; pause must be 0–2 seconds.')\n    if blend_voice not in (None, 'None') and (blend_voice not in VOICE_CHOICES or blend_voice[0] != voice[0]):\n        raise ValueError('Blend voices must use the same language/accent.')\n    if not 0 <= float(blend) <= 1:\n        raise ValueError('Blend weight must be 0–1.')\n    selected = get_pipeline(voice[0])\n    effective_voice = voice\n    if blend_voice not in (None, 'None'):\n        effective_voice = selected.load_voice(voice) * (1 - float(blend)) + selected.load_voice(blend_voice) * float(blend)\n    chunks, subtitles, cursor = [], [], 0.0\n    for part in split_text(text):\n        part_audio = []\n        for item in selected(part, voice=effective_voice, speed=float(speed)):\n            audio = item[2] if isinstance(item, tuple) and len(item) >= 3 else item.get('audio') if isinstance(item, dict) else getattr(item, 'audio', None)\n            if audio is None:\n                raise RuntimeError(f'Unsupported Kokoro output type: {type(item)}')\n            if hasattr(audio, 'detach'):\n                audio = audio.detach().cpu().numpy()\n            array = np.asarray(audio, dtype=np.float32).reshape(-1)\n            if array.size == 0 or not np.isfinite(array).all():\n                raise RuntimeError('Invalid Kokoro audio.')\n            part_audio.append(array)\n        if not part_audio:\n            raise RuntimeError('Kokoro returned no audio.')\n        joined = np.concatenate(part_audio)\n        if chunks:\n            chunks.append(np.zeros(round(float(pause) * SAMPLE_RATE), dtype=np.float32))\n            cursor += float(pause)\n        start = cursor\n        chunks.append(joined)\n        cursor += len(joined) / SAMPLE_RATE\n        subtitles.append(f'{len(subtitles)+1}\\n{timestamp(start)} --> {timestamp(cursor)}\\n{part}\\n')\n    if not chunks:\n        raise RuntimeError('Kokoro returned no audio.')\n    wav = OUT / f'kokoro_{time.time_ns()}.wav'\n    sf.write(wav, np.concatenate(chunks), SAMPLE_RATE)\n    result = export_audio(wav, format, pitch, normalize)\n    Path(result).with_suffix('.srt').write_text('\\n'.join(subtitles), encoding='utf-8')\n    save_metadata(result, dict(text=text, voice=voice, blend_voice=blend_voice, blend=blend, speed=speed, pause=pause, pitch=pitch, normalize=normalize, duration=cursor, subtitles='chunk-level approximate timing'))\n    return result\n\n\ndef ui_generate(*args):\n    try:\n        result = generate_audio(*args)\n        return result, result, str(Path(result).with_suffix('.srt')), f'{sf.info(result).duration:.2f} seconds'\n    except Exception as e:\n        raise gr.Error(str(e)) from e\n\n\ndef batch_generate(text, *settings):\n    lines = [line.strip() for line in (text or '').splitlines() if line.strip()]\n    if not lines or len(lines) > 50:\n        raise gr.Error('Enter 1–50 nonempty lines, one clip per line.')\n    try:\n        files = [generate_audio(line, *settings) for line in lines]\n        return package_files(files, OUT, 'kokoro_batch')\n    except Exception as e:\n        raise gr.Error(str(e)) from e\n\n\ndef build_demo():\n    with gr.Blocks(title='AI Music OS — Kokoro v7') as demo:\n        gr.Markdown('## Kokoro TTS — 41 voices · 7 languages')\n        with gr.Row():\n            language = gr.Dropdown(list(LANGUAGES), value='American English', label='Language / accent', min_width=240)\n            voice = gr.Dropdown(voices_for_language('American English'), value='af_heart', label='Voice', min_width=240)\n        with gr.Row():\n            speed = gr.Slider(0.5, 2, value=1, step=0.05, label='Speech speed', min_width=240)\n            pause = gr.Slider(0, 2, value=0.15, step=0.05, label='Pause between chunks (seconds)', min_width=240)\n        with gr.Accordion('Voice blend & export', open=False):\n            second = gr.Dropdown(['None'] + voices_for_language('American English'), value='None', label='Second voice')\n            blend = gr.Slider(0, 1, value=0.5, step=0.05, label='Second voice weight')\n            format = gr.Radio(['WAV', 'MP3', 'FLAC'], value='WAV', label='Export format')\n            pitch = gr.Slider(-6, 6, value=0, step=0.5, label='Pitch shift (semitones)')\n            normalize = gr.Checkbox(label='Normalize loudness', value=False)\n        settings = [voice, speed, pause, second, blend, format, pitch, normalize]\n        with gr.Tab('Speak'):\n            text = gr.Textbox(lines=8, label='Text', value='Hello. Welcome to Music OS.')\n            upload = gr.File(file_types=['.txt'], label='Import text', type='filepath')\n            gen = gr.Button('Generate speech', variant='primary')\n            audio = gr.Audio(type='filepath', label='Speech')\n            download = gr.File(label='Download audio')\n            srt = gr.File(label='Subtitles (approximate chunk timing)')\n            info = gr.Textbox(label='Duration')\n        with gr.Tab('Batch clips'):\n            batch = gr.Textbox(lines=10, label='One clip per line')\n            batch_button = gr.Button('Generate batch', variant='primary')\n            archive = gr.File(label='Clips, settings & subtitles ZIP')\n        with gr.Tab('Saved outputs'):\n            refresh = gr.Button('Refresh')\n            history = gr.Files(label='Recent downloads')\n        language.change(change_language, language, [voice, second])\n        upload.change(read_text_file, upload, text)\n        gen.click(ui_generate, [text] + settings, [audio, download, srt, info])\n        batch_button.click(batch_generate, [batch] + settings, archive)\n        refresh.click(lambda: recent_outputs(OUT), outputs=history)\n    return demo\n\n\nif __name__ == '__main__':\n    print('Kokoro smoke test:', generate_audio('Hello. This is Music OS.', 'af_heart'))\n    build_demo().queue(default_concurrency_limit=1).launch(share=True, server_name='0.0.0.0', server_port=7866, allowed_paths=[str(OUT)])\n", encoding='utf-8')
ENV['AI_MUSIC_OUT'] = str(OUT)
ENV['AI_MUSIC_VOICE_BANK'] = str(VOICE_BANK)
COMMAND = [str(PY), str(APP)]


In [ ]:
# Launch in the isolated environment; rerunning safely replaces the previous process.
if 'proc' in globals() and proc.poll() is None:
    proc.terminate()
    try: proc.wait(timeout=20)
    except subprocess.TimeoutExpired: proc.kill(); proc.wait()
LAUNCH_LOG = OUT / 'launch_v6.log'
with LAUNCH_LOG.open('w') as log:
    proc = subprocess.Popen(COMMAND, cwd=str(OUT), env=ENV, stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(900):
    time.sleep(2)
    text = LAUNCH_LOG.read_text(errors='replace')
    if proc.poll() is not None:
        print(text[-18000:])
        raise RuntimeError(f'Application exited. Full log: {LAUNCH_LOG}')
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match: url = match.group(0); break
if url is None:
    print(LAUNCH_LOG.read_text(errors='replace')[-6000:])
    raise TimeoutError(f'No share URL after 30 minutes; inspect {LAUNCH_LOG}. Rerun this cell to retry.')
print('Open Music OS:', url)
print('Keep this Colab session open. Logs:', LAUNCH_LOG)


In [ ]:
# Inspect the application log at any time.
print(LAUNCH_LOG.read_text(errors='replace')[-15000:])
